In [ ]:
import torch
import torchaudio
import torchaudio.functional as F
import sounddevice as sd
import soundfile as sf
import numpy as np
import threading
import queue
import time
import signal
import sys

# =====================================================
# REALTIME SOURCE SEPARATION PRO v2
# ConvTasNet + robust multithread pipeline
# =====================================================

# ---------------- CONFIG ----------------
MIC_SR = 16000          # microphone sample rate
MODEL_SR = 8000         # ConvTasNet expected sample rate
CHANNELS = 1

CHUNK_MS = 1000           # safer than 20ms
WINDOW_MS = 2000         # lower latency than 800ms

QUEUE_MAX = 256

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# save names
OUT1 = "source1.wav"
OUT2 = "source2.wav"

# ---------------------------------------

chunk_mic = int(MIC_SR * CHUNK_MS / 1000)
chunk_model = int(MODEL_SR * CHUNK_MS / 1000)
window_model = int(MODEL_SR * WINDOW_MS / 1000)

print("=" * 60)
print("Realtime Separator PRO v2")
print("Device:", DEVICE)
print("Mic SR:", MIC_SR)
print("Model SR:", MODEL_SR)
print("Chunk:", CHUNK_MS, "ms")
print("Window:", WINDOW_MS, "ms")
print("=" * 60)

# =====================================================
# MODEL LOAD
# =====================================================
print("Loading ConvTasNet...")

bundle = torchaudio.pipelines.CONVTASNET_BASE_LIBRI2MIX
model = bundle.get_model().to(DEVICE)
model.eval()

print("Model loaded.")

# =====================================================
# GLOBALS
# =====================================================
audio_q = queue.Queue(maxsize=QUEUE_MAX)

running = True
capture_chunks = 0
processed_chunks = 0
dropped_chunks = 0

source1_all = []
source2_all = []

buffer = torch.zeros(1, 1, window_model, device=DEVICE)

proc_times = []

# =====================================================
# AUDIO CALLBACK (THREAD SAFE)
# =====================================================
def callback(indata, frames, time_info, status):
    global capture_chunks, dropped_chunks

    if status:
        print("Audio status:", status)

    mono = indata[:, 0].copy()

    try:
        audio_q.put_nowait(mono)
        capture_chunks += 1
    except queue.Full:
        dropped_chunks += 1


# =====================================================
# INFERENCE THREAD
# =====================================================
def worker():
    global running, processed_chunks, buffer

    last_log = time.time()

    while running or not audio_q.empty():

        try:
            chunk = audio_q.get(timeout=0.2)
        except queue.Empty:
            continue

        t0 = time.time()

        # numpy -> tensor
        x = torch.tensor(chunk).float().unsqueeze(0).unsqueeze(0)

        # resample mic -> model
        x = F.resample(x, MIC_SR, MODEL_SR).to(DEVICE)
        rms = torch.sqrt(torch.mean(x**2) + 1e-8)
        target = 0.08
        gain = target / (rms + 1e-8)
        gain = torch.clamp(gain, 0.5, 4.0)

        x = x * gain
        x = torch.clamp(x, -1.0, 1.0)

        # rolling buffer
        buffer = torch.roll(buffer, -chunk_model, dims=-1)
        buffer[:, :, -chunk_model:] = x

        # inference
        with torch.no_grad():
            sep = model(buffer)

        out1 = sep[0, 0, -chunk_model:].detach().cpu().numpy()
        out2 = sep[0, 1, -chunk_model:].detach().cpu().numpy()

        source1_all.append(out1)
        source2_all.append(out2)

        processed_chunks += 1

        t1 = time.time()
        proc_times.append(t1 - t0)

        # light logging every second
        if time.time() - last_log > 1.0:
            avg = np.mean(proc_times[-50:]) if proc_times else 0
            print(
                f"Captured={capture_chunks} | "
                f"Processed={processed_chunks} | "
                f"Queue={audio_q.qsize()} | "
                f"Dropped={dropped_chunks} | "
                f"AvgChunkProc={avg*1000:.1f} ms"
            )
            last_log = time.time()


# =====================================================
# STOP HANDLER
# =====================================================
def stop_handler(sig, frame):
    global running
    print("\nStopping requested...")
    running = False


signal.signal(signal.SIGINT, stop_handler)

# =====================================================
# START THREAD
# =====================================================
thread = threading.Thread(target=worker, daemon=True)
thread.start()

# =====================================================
# START MIC STREAM
# =====================================================
print("Opening microphone...")
print("Speak / play audio from phone.")
print("Press CTRL+C to stop.\n")

t_global0 = time.time()

with sd.InputStream(
    samplerate=MIC_SR,
    blocksize=chunk_mic,
    channels=1,
    callback=callback
):
    while running:
        time.sleep(0.1)

# =====================================================
# WAIT THREAD FINISH
# =====================================================
print("Finishing remaining queued audio...")
thread.join()

t_global1 = time.time()

# =====================================================
# SAVE FILES
# =====================================================
print("Saving WAV files...")

if len(source1_all) == 0:
    print("No audio processed.")
    sys.exit()

s1 = np.concatenate(source1_all)
s2 = np.concatenate(source2_all)

# normalize
s1 /= np.max(np.abs(s1)) + 1e-9
s2 /= np.max(np.abs(s2)) + 1e-9

sf.write(OUT1, s1, MODEL_SR)
sf.write(OUT2, s2, MODEL_SR)

# =====================================================
# METRICS
# =====================================================
audio_duration = len(s1) / MODEL_SR
proc_total = t_global1 - t_global0
rtf = proc_total / audio_duration if audio_duration > 0 else 999

corr = np.corrcoef(s1, s2)[0, 1]

print("\n" + "=" * 60)
print("FINAL RESULTS")
print("=" * 60)
print(f"Captured chunks : {capture_chunks}")
print(f"Processed chunks: {processed_chunks}")
print(f"Dropped chunks  : {dropped_chunks}")
print(f"Separated audio : {audio_duration:.2f} s")
print(f"Wall time       : {proc_total:.2f} s")
print(f"RTF             : {rtf:.4f}")
print(f"Correlation     : {corr:.4f}")
print(f"Saved           : {OUT1}")
print(f"Saved           : {OUT2}")

if rtf < 1:
    print("Realtime capable ✔️")
else:
    print("Too slow for strict realtime ✖️")

print("=" * 60)